# AMIE: Assimilative Mapping of Ionospheric Electrodynamics
## Richmond & Kamide (1988) — Implementation / 구현

This notebook implements the core concepts of the AMIE technique:
1. **Part 1**: Basis functions for high-latitude ionosphere / 고위도 전리층 기저 함수
2. **Part 2**: Forward model — potential → E, J, ΔB / 순방향 모델
3. **Part 3**: Optimal linear estimation from scratch / 최적 선형 추정 직접 구현
4. **Part 4**: Synthetic AMIE with simulated observations / 합성 관측을 이용한 AMIE 시뮬레이션
5. **Part 5**: Effect of observation density on error / 관측 밀도에 따른 오차 변화
6. **Part 6**: Conductance modification / 전도도 수정
7. **Part 7**: Comparison with KRM-like approach / KRM 방식과의 비교
8. **Part 8**: Connection to modern data assimilation / 현대 데이터 동화와의 연결

이 노트북은 Richmond & Kamide (1988)의 핵심 알고리즘을 교육 목적으로 재구현합니다.
This notebook reimplements the core algorithms of Richmond & Kamide (1988) for educational purposes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.special import lpmv
from scipy.linalg import solve, inv
from numpy.linalg import norm

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 11

# Constants
R_E = 6371.0   # Earth radius in km
R_I = 6481.0   # Ionospheric current shell radius in km (110 km altitude)
MU_0 = 4 * np.pi * 1e-7  # Permeability of free space

print("Libraries loaded successfully.")

## Part 1: Basis Functions for High-Latitude Ionosphere / 고위도 전리층 기저 함수

Richmond & Kamide used generalized associated Legendre functions as basis functions.
For this educational implementation, we use a simplified 2D polar coordinate system
(colatitude θ, longitude φ) with Fourier-Legendre basis functions.

AMIE의 기저 함수는 전위 $\Phi$를 직교정규 함수들의 선형 결합으로 표현합니다:

$$\Phi(\theta, \phi) = \sum_{i} a_i \Phi_i(\theta, \phi)$$

여기서 각 $\Phi_i$는 위도 방향 함수 $P(\theta)$와 경도 방향 함수 $f_m(\phi)$의 곱입니다.

We simplify the problem to a flat polar cap geometry for clarity,
using radial basis functions $R_n(r)$ and azimuthal harmonics $f_m(\phi)$.

In [ ]:
def make_polar_grid(n_r=50, n_phi=72):
    """Create a polar grid for the high-latitude ionosphere.

    Args:
        n_r: Number of radial points (colatitude from pole to boundary).
        n_phi: Number of azimuthal points (MLT).

    Returns:
        Tuple of (r, phi, R, PHI) where r, phi are 1D and R, PHI are 2D meshgrids.
    """
    theta_0 = 40.0  # Colatitude boundary in degrees (50° magnetic latitude)
    r = np.linspace(0.01, theta_0, n_r)  # Colatitude in degrees
    phi = np.linspace(0, 2 * np.pi, n_phi, endpoint=False)
    R, PHI = np.meshgrid(r, phi, indexing='ij')
    return r, phi, R, PHI


def basis_functions_2d(R, PHI, n_max=5, m_max=4):
    """Generate simplified 2D basis functions on polar cap.

    Uses Bessel-like radial functions with azimuthal harmonics,
    analogous to the associated Legendre functions in the paper.

    Args:
        R: 2D colatitude grid (degrees).
        PHI: 2D longitude grid (radians).
        n_max: Maximum radial order.
        m_max: Maximum azimuthal wave number.

    Returns:
        List of (basis_function_2D, label_string) tuples.
    """
    theta_0 = R.max()
    r_norm = R / theta_0  # Normalized to [0, 1]

    basis_list = []
    for m in range(0, m_max + 1):
        for n in range(max(1, m), n_max + 1):
            # Radial part: r^m * (1 - r^2)^(n-m) for smoothness
            # This mimics the Legendre function behavior
            radial = r_norm**m * np.cos(n * np.pi * r_norm / 2)

            if m == 0:
                bf = radial
                label = f"n={n}, m=0"
                basis_list.append((bf, label))
            else:
                # cos(m*phi) component
                bf_cos = radial * np.cos(m * PHI) * np.sqrt(2)
                label_cos = f"n={n}, m={m} (cos)"
                basis_list.append((bf_cos, label_cos))

                # sin(m*phi) component
                bf_sin = radial * np.sin(m * PHI) * np.sqrt(2)
                label_sin = f"n={n}, m={m} (sin)"
                basis_list.append((bf_sin, label_sin))

    return basis_list


# Create grid and basis functions
r, phi, R, PHI = make_polar_grid()
basis_list = basis_functions_2d(R, PHI, n_max=5, m_max=3)

print(f"Number of basis functions: {len(basis_list)}")
print(f"Grid: {R.shape[0]} radial × {R.shape[1]} azimuthal points")
print(f"Polar cap boundary: {R.max():.0f}° colatitude (= {90 - R.max():.0f}° magnetic latitude)")

In [ ]:
# Visualize a selection of basis functions (cf. Figure 2 in the paper)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), subplot_kw={'projection': 'polar'})

for idx, ax in enumerate(axes.flat):
    if idx >= len(basis_list):
        ax.set_visible(False)
        continue
    bf, label = basis_list[idx]
    r_rad = np.deg2rad(R)
    c = ax.pcolormesh(PHI, r_rad, bf, cmap='RdBu_r', shading='auto')
    ax.set_title(label, fontsize=10, pad=12)
    ax.set_ylim(0, np.deg2rad(40))
    ax.set_yticks(np.deg2rad([10, 20, 30, 40]))
    ax.set_yticklabels(['80°', '70°', '60°', '50°'], fontsize=7)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(['00', '06', '12', '18'], fontsize=8)

fig.suptitle('Basis Functions $\Phi_i(\\theta, \phi)$ for High-Latitude Ionosphere\n'
             '고위도 전리층 기저 함수 (cf. Richmond & Kamide Fig. 2)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 2: Forward Model — Potential → E, J, ΔB / 순방향 모델

Given a set of coefficients $\{a_i\}$ and conductances $\Sigma_P$, $\Sigma_H$, we compute:

1. **Electric potential / 전위**: $\Phi = \sum_i a_i \Phi_i$
2. **Electric field / 전기장**: $\mathbf{E} = -\nabla\Phi$
3. **Ionospheric current / 전리층 전류**: $\mathbf{I} = \Sigma_P \mathbf{E} + \Sigma_H (\hat{b} \times \mathbf{E})$
4. **Field-aligned current / FAC**: $J_\parallel = -\nabla \cdot \mathbf{I}$

전위 $\Phi$에서 시작하여 모든 전기역학 변수를 물리법칙으로 유도하는 과정입니다.
This is the process of deriving all electrodynamic variables from the potential $\Phi$ using physical laws.

In [ ]:
def create_synthetic_potential(R, PHI, pattern='two_cell'):
    """Create a synthetic ionospheric potential pattern.

    Args:
        R: 2D colatitude grid (degrees).
        PHI: 2D longitude grid (radians).
        pattern: Type of convection pattern ('two_cell' or 'substorm').

    Returns:
        2D potential in Volts.
    """
    theta_0 = R.max()
    r_norm = R / theta_0

    if pattern == 'two_cell':
        # Classic two-cell convection (Dungey cycle)
        # Dawn cell (positive) and dusk cell (negative)
        phi_shift = np.pi / 2  # Dawn-dusk orientation
        # Potential peaks at auroral zone (~70° = 20° colat)
        radial_env = r_norm * np.exp(-((r_norm - 0.5)**2) / 0.08)
        potential = 30000 * radial_env * np.sin(PHI - phi_shift)  # ±30 kV
    elif pattern == 'substorm':
        # Substorm: enhanced nightside with Harang discontinuity
        radial_env = r_norm * np.exp(-((r_norm - 0.5)**2) / 0.08)
        potential = 30000 * radial_env * np.sin(PHI - np.pi / 2)
        # Add nightside enhancement (around midnight, phi ~ pi)
        nightside = 15000 * np.exp(-((PHI - np.pi)**2) / 0.3) * \
                    np.exp(-((r_norm - 0.6)**2) / 0.05)
        potential += nightside

    return potential


def compute_electric_field(potential, R, PHI):
    """Compute electric field E = -grad(Phi) on polar grid.

    Args:
        potential: 2D potential array (V).
        R: 2D colatitude grid (degrees).
        PHI: 2D longitude grid (radians).

    Returns:
        Tuple of (E_theta, E_phi) components in V/m.
    """
    dr = np.deg2rad(np.mean(np.diff(R[:, 0]))) * R_I * 1e3  # meters
    dphi = np.mean(np.diff(PHI[0, :])) * R_I * 1e3  # meters at equator

    # E_theta = -dPhi/d(R_I * theta)
    E_theta = -np.gradient(potential, dr, axis=0)

    # E_phi = -1/(R_I*sin(theta)) * dPhi/dphi
    sin_theta = np.sin(np.deg2rad(R))
    sin_theta = np.maximum(sin_theta, 0.01)  # Avoid division by zero at pole
    E_phi = -np.gradient(potential, dphi, axis=1) / sin_theta

    return E_theta, E_phi


def compute_currents(E_theta, E_phi, Sigma_P, Sigma_H):
    """Compute height-integrated ionospheric currents.

    I = Sigma_P * E + Sigma_H * (b_hat x E)

    For downward B (northern hemisphere), b_hat x E rotates E by -90°:
    (b x E)_theta = E_phi,  (b x E)_phi = -E_theta

    Args:
        E_theta: Theta component of E field.
        E_phi: Phi component of E field.
        Sigma_P: Pedersen conductance (S).
        Sigma_H: Hall conductance (S).

    Returns:
        Tuple of (I_theta, I_phi) current components in A/m.
    """
    I_theta = Sigma_P * E_theta + Sigma_H * E_phi
    I_phi = Sigma_P * E_phi - Sigma_H * E_theta
    return I_theta, I_phi


# Create synthetic two-cell convection potential
potential_true = create_synthetic_potential(R, PHI, 'two_cell')

# Uniform conductance model
Sigma_P = 5.0   # Pedersen conductance in Siemens
Sigma_H = 10.0  # Hall conductance in Siemens

# Compute derived quantities
E_theta, E_phi = compute_electric_field(potential_true, R, PHI)
I_theta, I_phi = compute_currents(E_theta, E_phi, Sigma_P, Sigma_H)
E_mag = np.sqrt(E_theta**2 + E_phi**2)

print(f"Potential range: {potential_true.min()/1000:.1f} to {potential_true.max()/1000:.1f} kV")
print(f"Cross-polar cap potential drop: {(potential_true.max() - potential_true.min())/1000:.1f} kV")
print(f"Max electric field: {E_mag.max()*1000:.1f} mV/m")

In [ ]:
# Visualize the forward model: Potential → E → J
# (cf. Figure 4 in the paper — equivalent current and electric potential)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': 'polar'})
r_rad = np.deg2rad(R)

# Electric Potential
ax = axes[0]
levels = np.linspace(-30000, 30000, 13)
c0 = ax.contourf(PHI, r_rad, potential_true, levels=levels, cmap='RdBu_r')
ax.contour(PHI, r_rad, potential_true, levels=levels, colors='k', linewidths=0.5)
ax.set_title('Electric Potential $\Phi$ (V)\n전위', fontsize=11)
plt.colorbar(c0, ax=ax, shrink=0.8, label='V')

# Electric Field Magnitude
ax = axes[1]
c1 = ax.pcolormesh(PHI, r_rad, E_mag * 1000, cmap='hot_r', shading='auto',
                    vmin=0, vmax=E_mag.max() * 1000)
# Overlay E-field vectors (subsampled)
skip = (5, 6)
ax.quiver(PHI[::skip[0], ::skip[1]], r_rad[::skip[0], ::skip[1]],
          E_phi[::skip[0], ::skip[1]], E_theta[::skip[0], ::skip[1]],
          color='blue', scale=0.3, alpha=0.7)
ax.set_title('Electric Field $|\\mathbf{E}|$ (mV/m)\n전기장', fontsize=11)
plt.colorbar(c1, ax=ax, shrink=0.8, label='mV/m')

# Ionospheric Current
ax = axes[2]
I_mag = np.sqrt(I_theta**2 + I_phi**2)
c2 = ax.pcolormesh(PHI, r_rad, I_mag * 1000, cmap='YlOrRd', shading='auto')
ax.quiver(PHI[::skip[0], ::skip[1]], r_rad[::skip[0], ::skip[1]],
          I_phi[::skip[0], ::skip[1]], I_theta[::skip[0], ::skip[1]],
          color='navy', scale=2.0, alpha=0.7)
ax.set_title('Ionospheric Current $|\\mathbf{I}|$ (mA/m)\n전리층 전류', fontsize=11)
plt.colorbar(c2, ax=ax, shrink=0.8, label='mA/m')

for ax in axes:
    ax.set_ylim(0, np.deg2rad(40))
    ax.set_yticks(np.deg2rad([10, 20, 30, 40]))
    ax.set_yticklabels(['80°', '70°', '60°', '50°'], fontsize=8)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(['00', '06', '12', '18'], fontsize=9)

fig.suptitle('Forward Model: $\Phi \\rightarrow \\mathbf{E} \\rightarrow \\mathbf{I}$\n'
             '순방향 모델: 전위에서 전기장과 전류 도출',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 3: Optimal Linear Estimation from Scratch / 최적 선형 추정 직접 구현

This is the mathematical core of AMIE. We implement the Gauss-Markov optimal estimation:

$$\hat{\mathbf{a}} = \mathbf{A} \boldsymbol{\eta}, \quad \mathbf{A} = \mathbf{C}_a \mathbf{D}^T (\mathbf{D} \mathbf{C}_a \mathbf{D}^T + \mathbf{C}_v)^{-1}$$

AMIE의 수학적 핵심을 구현합니다:
- **$\mathbf{D}$** (관측 연산자): 기저 함수 계수를 관측 가능량으로 변환 / converts coefficients to observables
- **$\mathbf{C}_a$** (계수 공분산): 전기역학 필드의 통계적 변동성 / statistical variability
- **$\mathbf{C}_v$** (오차 공분산): 관측 오차 + 절단 오차 / observation + truncation error
- **$\boldsymbol{\eta}$** (관측 편차): 관측값 - 배경 모델 예측값 / observation - background prediction

In [ ]:
class AMIESolver:
    """Simplified AMIE solver implementing optimal linear estimation.

    This class demonstrates the core mathematical algorithm of
    Richmond & Kamide (1988) on a 2D polar cap grid.

    Attributes:
        R: 2D colatitude grid (degrees).
        PHI: 2D longitude grid (radians).
        basis_list: List of (basis_function_2D, label) tuples.
        I: Number of basis functions.
    """

    def __init__(self, R, PHI, basis_list):
        self.R = R
        self.PHI = PHI
        self.basis_list = basis_list
        self.I = len(basis_list)

    def build_observation_operator(self, obs_locations, obs_type='potential'):
        """Build observation operator matrix D (J x I).

        Maps basis function coefficients to predicted observation values.

        Args:
            obs_locations: List of (r_idx, phi_idx) tuples for observation points.
            obs_type: Type of observation ('potential' or 'electric_field').

        Returns:
            D matrix of shape (J, I).
        """
        J = len(obs_locations)
        D = np.zeros((J, self.I))

        for j, (ri, pi) in enumerate(obs_locations):
            for i, (bf, _) in enumerate(self.basis_list):
                if obs_type == 'potential':
                    D[j, i] = bf[ri, pi]
                elif obs_type == 'electric_field':
                    # Use gradient of basis function (simplified)
                    dr = max(1, 1)
                    if ri + dr < bf.shape[0]:
                        D[j, i] = -(bf[ri + dr, pi] - bf[ri, pi])
                    else:
                        D[j, i] = -(bf[ri, pi] - bf[ri - dr, pi])

        return D

    def build_Ca(self, sigma_a=1.0, power_law_exp=2):
        """Build coefficient covariance matrix C_a.

        Following the paper, C_a is diagonal with entries that decrease
        with increasing spatial frequency (higher-order basis functions
        have smaller variance — large-scale structures dominate).

        Args:
            sigma_a: Overall amplitude scale.
            power_law_exp: Power law exponent for spectral decay.

        Returns:
            C_a matrix of shape (I, I).
        """
        Ca = np.zeros((self.I, self.I))
        for i, (_, label) in enumerate(self.basis_list):
            # Extract n and m from label
            parts = label.split(',')
            n = int(parts[0].split('=')[1])
            m_str = parts[1].strip().split('=')[1].split()[0]
            m = int(m_str)
            # Variance decreases with spatial order (Eq. 44-48 in paper)
            order = max(n, 1) + max(m, 0)
            Ca[i, i] = sigma_a**2 / (order ** power_law_exp)

        return Ca

    def build_Cv(self, J, sigma_obs=0.05, sigma_trunc=0.02):
        """Build error covariance matrix C_v.

        C_v = <v_j v_j'> = observation error + truncation error (Eq. 57-58)

        Args:
            J: Number of observations.
            sigma_obs: Observation error standard deviation.
            sigma_trunc: Truncation error standard deviation.

        Returns:
            C_v matrix of shape (J, J).
        """
        total_var = sigma_obs**2 + sigma_trunc**2
        Cv = np.eye(J) * total_var
        return Cv

    def solve(self, D, Ca, Cv, eta):
        """Solve for optimal coefficients using Gauss-Markov theorem.

        a_hat = A * eta, where A = Ca D^T (D Ca D^T + Cv)^{-1}  (Eq. 35)

        Args:
            D: Observation operator matrix (J, I).
            Ca: Coefficient covariance (I, I).
            Cv: Error covariance (J, J).
            eta: Observation deviations vector (J,).

        Returns:
            Tuple of (a_hat, A) — estimated coefficients and weight matrix.
        """
        # A = Ca D^T (D Ca D^T + Cv)^{-1}
        DCa = D @ Ca
        S = DCa @ D.T + Cv  # (J, J) matrix
        A = Ca @ D.T @ inv(S)  # (I, J) weight matrix
        a_hat = A @ eta
        return a_hat, A

    def reconstruct(self, a_hat):
        """Reconstruct the potential field from estimated coefficients.

        Phi = sum_i a_hat_i * Phi_i  (Eq. 15 applied)

        Args:
            a_hat: Estimated coefficients (I,).

        Returns:
            Reconstructed potential field (2D array).
        """
        potential = np.zeros_like(self.R)
        for i, (bf, _) in enumerate(self.basis_list):
            potential += a_hat[i] * bf
        return potential

    def compute_error_map(self, A, D, Ca):
        """Compute the estimation error at each grid point.

        Error^2 = <epsilon^2> - <epsilon_hat^2>  (Eq. 56)
        This is evaluated by comparing the total variability to
        the estimated variability.

        Args:
            A: Weight matrix (I, J).
            D: Observation operator (J, I).
            Ca: Coefficient covariance (I, I).

        Returns:
            2D error map.
        """
        # Total variability from Ca
        total_var = np.zeros_like(self.R)
        for i, (bf, _) in enumerate(self.basis_list):
            total_var += Ca[i, i] * bf**2

        # Estimated variability: uses A*D*Ca structure
        AD = A @ D  # (I, I)
        estimated_var = np.zeros_like(self.R)
        for i, (bf_i, _) in enumerate(self.basis_list):
            for k, (bf_k, _) in enumerate(self.basis_list):
                coeff = 0
                for l in range(self.I):
                    coeff += AD[i, l] * Ca[l, k]
                estimated_var += coeff * bf_i * bf_k

        error_map = np.sqrt(np.maximum(total_var - estimated_var, 0))
        return error_map


solver = AMIESolver(R, PHI, basis_list)
print(f"AMIE solver initialized with {solver.I} basis functions.")

## Part 4: Synthetic AMIE with Simulated Observations / 합성 관측을 이용한 AMIE 시뮬레이션

We simulate the full AMIE workflow:
1. Create a "true" potential pattern (known but hidden)
2. Sample it at sparse observation locations + add noise
3. Apply AMIE optimal estimation to recover the global pattern
4. Compare recovered vs. true pattern

전체 AMIE 작업 흐름을 시뮬레이션합니다:
1. "실제" 전위 패턴 생성 (알려져 있지만 숨겨진 상태)
2. 희소 관측 지점에서 샘플링 + 노이즈 추가
3. AMIE 최적 추정 적용으로 전구적 패턴 복원
4. 복원된 패턴과 실제 패턴 비교

In [ ]:
np.random.seed(42)

# --- Step 1: Project true potential onto basis functions ---
# Find the "true" coefficients by least-squares projection
n_r, n_phi_grid = R.shape
phi_flat = basis_list[0][0].flatten()  # Just for shape

# Build matrix of basis function values at all grid points
n_grid = n_r * n_phi_grid
B_matrix = np.zeros((n_grid, solver.I))
for i, (bf, _) in enumerate(basis_list):
    B_matrix[:, i] = bf.flatten()

# Project true potential onto basis functions
pot_flat = potential_true.flatten()
a_true, _, _, _ = np.linalg.lstsq(B_matrix, pot_flat, rcond=None)

# Verify projection quality
pot_projected = solver.reconstruct(a_true)
proj_error = np.sqrt(np.mean((pot_projected - potential_true)**2))
print(f"Projection RMSE: {proj_error:.0f} V "
      f"({proj_error/np.std(potential_true)*100:.1f}% of std)")

# --- Step 2: Create sparse observation network ---
# Simulate magnetometer chain locations (like Table 1 in the paper)
obs_locations = []
obs_labels = []

# Alaska chain (high latitude, ~65-80° = 10-25° colat)
for colat_deg in [12, 16, 20, 25, 30]:
    ri = int(colat_deg / 40 * (n_r - 1))
    pi = int(262 / 360 * n_phi_grid) % n_phi_grid  # ~262° longitude
    obs_locations.append((ri, pi))
    obs_labels.append(f'AK {90-colat_deg}°')

# Scandinavia chain
for colat_deg in [10, 15, 20, 25, 30, 35]:
    ri = int(colat_deg / 40 * (n_r - 1))
    pi = int(105 / 360 * n_phi_grid) % n_phi_grid  # ~105° longitude
    obs_locations.append((ri, pi))
    obs_labels.append(f'SC {90-colat_deg}°')

# Canada chain
for colat_deg in [14, 18, 22, 28, 33]:
    ri = int(colat_deg / 40 * (n_r - 1))
    pi = int(300 / 360 * n_phi_grid) % n_phi_grid
    obs_locations.append((ri, pi))
    obs_labels.append(f'CA {90-colat_deg}°')

# Greenland chain
for colat_deg in [8, 13, 18, 23]:
    ri = int(colat_deg / 40 * (n_r - 1))
    pi = int(28 / 360 * n_phi_grid) % n_phi_grid
    obs_locations.append((ri, pi))
    obs_labels.append(f'GL {90-colat_deg}°')

J = len(obs_locations)
print(f"\nNumber of observation stations: {J}")
print(f"Observation chains: Alaska, Scandinavia, Canada, Greenland")

# --- Step 3: Sample observations + add noise ---
sigma_noise = 1500  # Noise in Volts (equivalent to ~27 nT magnetometer noise)
obs_true = np.array([potential_true[ri, pi] for ri, pi in obs_locations])
obs_noise = obs_true + np.random.randn(J) * sigma_noise

# Background model = zero (no prior knowledge of deviation)
eta = obs_noise  # Observation deviations from background (zero background)

print(f"Observation noise std: {sigma_noise} V")
print(f"True signal range: {obs_true.min():.0f} to {obs_true.max():.0f} V")

In [ ]:
# --- Step 4: Apply AMIE optimal estimation ---
D = solver.build_observation_operator(obs_locations, obs_type='potential')
Ca = solver.build_Ca(sigma_a=15000, power_law_exp=2)
Cv = solver.build_Cv(J, sigma_obs=sigma_noise, sigma_trunc=500)

# Solve
a_hat, A_weight = solver.solve(D, Ca, Cv, eta)
potential_amie = solver.reconstruct(a_hat)

# --- Step 5: Compare ---
rmse = np.sqrt(np.mean((potential_amie - potential_true)**2))
corr = np.corrcoef(potential_amie.flatten(), potential_true.flatten())[0, 1]
print(f"AMIE Reconstruction Quality:")
print(f"  RMSE: {rmse:.0f} V ({rmse/np.std(potential_true)*100:.1f}% of std)")
print(f"  Correlation: {corr:.4f}")
print(f"  Cross-polar cap potential: True={potential_true.max()-potential_true.min():.0f} V, "
      f"AMIE={potential_amie.max()-potential_amie.min():.0f} V")

# --- Visualize comparison ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': 'polar'})
r_rad = np.deg2rad(R)

vmax = max(abs(potential_true.max()), abs(potential_amie.max()))
levels = np.linspace(-vmax, vmax, 15)

# True potential
ax = axes[0]
c = ax.contourf(PHI, r_rad, potential_true, levels=levels, cmap='RdBu_r')
ax.contour(PHI, r_rad, potential_true, levels=levels, colors='k', linewidths=0.3)
ax.set_title('True Potential / 실제 전위', fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='V')

# AMIE reconstruction
ax = axes[1]
c = ax.contourf(PHI, r_rad, potential_amie, levels=levels, cmap='RdBu_r')
ax.contour(PHI, r_rad, potential_amie, levels=levels, colors='k', linewidths=0.3)
# Plot observation locations
for ri, pi in obs_locations:
    ax.plot(PHI[ri, pi], r_rad[ri, pi], 'k^', markersize=5)
ax.set_title(f'AMIE Reconstruction / AMIE 복원\n(corr={corr:.3f})', fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='V')

# Residual
ax = axes[2]
residual = potential_amie - potential_true
res_max = np.max(np.abs(residual))
levels_res = np.linspace(-res_max, res_max, 15)
c = ax.contourf(PHI, r_rad, residual, levels=levels_res, cmap='RdBu_r')
ax.set_title(f'Residual (AMIE - True) / 잔차\nRMSE={rmse:.0f} V', fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='V')

for ax in axes:
    ax.set_ylim(0, np.deg2rad(40))
    ax.set_yticks(np.deg2rad([10, 20, 30, 40]))
    ax.set_yticklabels(['80°', '70°', '60°', '50°'], fontsize=8)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(['00', '06', '12', '18'], fontsize=9)

fig.suptitle('AMIE Optimal Estimation: True vs. Reconstructed Potential\n'
             'AMIE 최적 추정: 실제 vs. 복원 전위 (cf. Fig. 4)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 5: Effect of Observation Density on Error / 관측 밀도에 따른 오차 변화

A key insight from the paper: **error decreases where observations are dense and
remains at the background variability level where observations are absent** (Fig. 7).

논문의 핵심 통찰: **오차는 관측이 밀집한 곳에서 감소하고, 관측이 없는 곳에서는
배경 변동성 수준에 머문다** (Fig. 7).

We demonstrate this by varying the number of observation stations and showing
how the reconstruction quality changes spatially.

In [ ]:
# Test with different observation densities
configs = [
    ("5 stations\n(sparse / 희소)", 5),
    ("20 stations\n(moderate / 적당)", 20),
    ("50 stations\n(dense / 밀집)", 50),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={'projection': 'polar'})

for col, (label, n_obs) in enumerate(configs):
    np.random.seed(42)

    # Generate random observation locations in auroral zone (15-35° colat)
    obs_locs = []
    for _ in range(n_obs):
        ri = np.random.randint(int(10/40*(n_r-1)), int(35/40*(n_r-1)))
        pi = np.random.randint(0, n_phi_grid)
        obs_locs.append((ri, pi))

    # Sample observations
    obs_vals = np.array([potential_true[ri, pi] for ri, pi in obs_locs])
    obs_noisy = obs_vals + np.random.randn(n_obs) * sigma_noise

    # AMIE solve
    D_test = solver.build_observation_operator(obs_locs, obs_type='potential')
    Cv_test = solver.build_Cv(n_obs, sigma_obs=sigma_noise, sigma_trunc=500)
    a_hat_test, A_test = solver.solve(D_test, Ca, Cv_test, obs_noisy)
    pot_test = solver.reconstruct(a_hat_test)

    # Actual error
    actual_error = np.abs(pot_test - potential_true)
    rmse_test = np.sqrt(np.mean(actual_error**2))
    corr_test = np.corrcoef(pot_test.flatten(), potential_true.flatten())[0, 1]

    # Plot reconstruction
    ax = axes[0, col]
    vmax = max(abs(potential_true.max()), abs(pot_test.max()))
    levels = np.linspace(-vmax, vmax, 15)
    c = ax.contourf(PHI, r_rad, pot_test, levels=levels, cmap='RdBu_r')
    for ri, pi in obs_locs:
        ax.plot(PHI[ri, pi], r_rad[ri, pi], 'k^', markersize=3)
    ax.set_title(f'{label}\ncorr={corr_test:.3f}', fontsize=10)
    plt.colorbar(c, ax=ax, shrink=0.7)

    # Plot error
    ax = axes[1, col]
    c = ax.pcolormesh(PHI, r_rad, actual_error / 1000, cmap='YlOrRd',
                       shading='auto', vmin=0, vmax=15)
    for ri, pi in obs_locs:
        ax.plot(PHI[ri, pi], r_rad[ri, pi], 'k^', markersize=3)
    ax.set_title(f'|Error| (kV)\nRMSE={rmse_test/1000:.1f} kV', fontsize=10)
    plt.colorbar(c, ax=ax, shrink=0.7, label='kV')

    for ax_row in [axes[0, col], axes[1, col]]:
        ax_row.set_ylim(0, np.deg2rad(40))
        ax_row.set_yticks(np.deg2rad([10, 20, 30, 40]))
        ax_row.set_yticklabels(['80°', '70°', '60°', '50°'], fontsize=7)
        ax_row.set_xticks(np.deg2rad([0, 90, 180, 270]))
        ax_row.set_xticklabels(['00', '06', '12', '18'], fontsize=8)

fig.suptitle('Effect of Observation Density on AMIE Reconstruction\n'
             '관측 밀도가 AMIE 복원에 미치는 영향 (cf. Fig. 7)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 6: Conductance Modification / 전도도 수정

The paper shows that conductance errors strongly affect the potential estimate.
AMIE modifies the conductance model in **log space** to keep conductances positive:

$$\ln(\Sigma_P / \Sigma_{P0}) = \sum_i \hat{s}_i L_i$$

논문에서 전도도 오차가 전위 추정에 크게 영향을 미치는 것을 보여줍니다.
AMIE는 전도도를 항상 양수로 유지하기 위해 **로그 공간**에서 수정합니다.

We demonstrate how incorrect conductance distorts the electric potential
and how AMIE's conductance correction improves the result.

In [ ]:
# Create a realistic non-uniform conductance model
# (cf. Figure 5 in the paper — Pedersen and Hall conductances)
theta_0 = R.max()
r_norm = R / theta_0

# True conductance: enhanced in auroral zone
Sigma_P_true = 3.0 + 8.0 * np.exp(-((r_norm - 0.55)**2) / 0.02)  # Auroral enhancement
# Add day-night asymmetry (higher on dayside due to solar EUV)
Sigma_P_true += 2.0 * np.cos(PHI)

Sigma_H_true = 2.0 * Sigma_P_true  # Hall/Pedersen ratio ~2

# Wrong conductance: uniform model (what you'd use without modification)
Sigma_P_wrong = np.ones_like(R) * 5.0
Sigma_H_wrong = np.ones_like(R) * 10.0

# Compute E fields from true potential
E_theta_true, E_phi_true = compute_electric_field(potential_true, R, PHI)

# Compute currents with TRUE conductance
I_theta_true, I_phi_true = compute_currents(E_theta_true, E_phi_true,
                                              Sigma_P_true, Sigma_H_true)

# If you try to invert E from J using WRONG conductance:
# E_estimated = J / Sigma_wrong (simplified scalar version)
E_theta_wrong = (Sigma_P_wrong * I_theta_true + Sigma_H_wrong * I_phi_true) / \
                (Sigma_P_wrong**2 + Sigma_H_wrong**2)
E_phi_wrong = (Sigma_P_wrong * I_phi_true - Sigma_H_wrong * I_theta_true) / \
              (Sigma_P_wrong**2 + Sigma_H_wrong**2)

# Apply log-space correction (Eq. 50-54 in paper)
# ln(Sigma_true / Sigma_wrong) = correction factor
log_correction = np.log(Sigma_P_true / Sigma_P_wrong)

# Corrected conductance
Sigma_P_corrected = Sigma_P_wrong * np.exp(log_correction)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': 'polar'})

ax = axes[0]
c = ax.pcolormesh(PHI, r_rad, Sigma_P_true, cmap='YlGnBu', shading='auto',
                   vmin=0, vmax=15)
ax.contour(PHI, r_rad, Sigma_P_true, levels=5, colors='k', linewidths=0.5)
ax.set_title('True $\Sigma_P$ (S)\n실제 Pedersen 전도도', fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='S')

ax = axes[1]
c = ax.pcolormesh(PHI, r_rad, Sigma_P_wrong, cmap='YlGnBu', shading='auto',
                   vmin=0, vmax=15)
ax.set_title('Assumed $\Sigma_P$ (S)\n가정된 Pedersen 전도도\n(uniform = 5 S)', fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='S')

ax = axes[2]
c = ax.pcolormesh(PHI, r_rad, log_correction, cmap='RdBu_r', shading='auto')
ax.contour(PHI, r_rad, log_correction, levels=7, colors='k', linewidths=0.5)
ax.set_title('Log Correction: $\ln(\Sigma_{true}/\Sigma_{wrong})$\n로그 보정 인자',
             fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='ln ratio')

for ax in axes:
    ax.set_ylim(0, np.deg2rad(40))
    ax.set_yticks(np.deg2rad([10, 20, 30, 40]))
    ax.set_yticklabels(['80°', '70°', '60°', '50°'], fontsize=8)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(['00', '06', '12', '18'], fontsize=9)

fig.suptitle('Conductance Model Modification (cf. Fig. 5)\n'
             '전도도 모델 수정 — AMIE는 관측을 이용하여 전도도를 보정',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"True Sigma_P range: {Sigma_P_true.min():.1f} – {Sigma_P_true.max():.1f} S")
print(f"Wrong Sigma_P (uniform): {Sigma_P_wrong[0,0]:.1f} S")
print(f"Log correction range: {log_correction.min():.2f} to {log_correction.max():.2f}")

## Part 7: Comparison with KRM-like Approach / KRM 방식과의 비교

The KRM technique (Kamide, Richmond, Matsushita, 1981) uses **only magnetometer data**
and requires a separate conductance assumption. It does not regularize with $\mathbf{C}_a$.

KRM 기법은 **자력계 데이터만** 사용하며 별도의 전도도 가정이 필요합니다.
$\mathbf{C}_a$에 의한 정규화가 없으므로, 관측이 부족한 지역에서 불안정할 수 있습니다.

We compare:
- **KRM-like**: Standard least-squares fit without regularization ($\mathbf{C}_a^{-1} = 0$)
- **AMIE**: Regularized optimal estimation with $\mathbf{C}_a$ constraint

In [ ]:
def solve_krm_like(D, Cv, eta):
    """KRM-like least-squares fit WITHOUT regularization.

    Minimizes ||D*a - eta||^2 / Cv only (no C_a constraint).
    This is equivalent to AMIE with C_a → infinity (no prior constraint).

    Args:
        D: Observation operator (J, I).
        Cv: Error covariance (J, J).
        eta: Observation deviations (J,).

    Returns:
        Estimated coefficients.
    """
    # a = (D^T Cv^{-1} D)^{-1} D^T Cv^{-1} eta
    Cv_inv = inv(Cv)
    DtCvD = D.T @ Cv_inv @ D
    # Add tiny regularization for numerical stability only
    DtCvD += np.eye(DtCvD.shape[0]) * 1e-10
    a_krm = solve(DtCvD, D.T @ Cv_inv @ eta)
    return a_krm


# Use the 20-station network from Part 4
D_20 = solver.build_observation_operator(obs_locations, obs_type='potential')
Cv_20 = solver.build_Cv(J, sigma_obs=sigma_noise, sigma_trunc=500)

# AMIE solution (with C_a regularization)
a_amie, _ = solver.solve(D_20, Ca, Cv_20, eta)
pot_amie = solver.reconstruct(a_amie)

# KRM-like solution (without C_a regularization)
a_krm = solve_krm_like(D_20, Cv_20, eta)
pot_krm = solver.reconstruct(a_krm)

# Compare
rmse_amie = np.sqrt(np.mean((pot_amie - potential_true)**2))
rmse_krm = np.sqrt(np.mean((pot_krm - potential_true)**2))
corr_amie = np.corrcoef(pot_amie.flatten(), potential_true.flatten())[0, 1]
corr_krm = np.corrcoef(pot_krm.flatten(), potential_true.flatten())[0, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': 'polar'})

# True
ax = axes[0]
vmax = max(abs(potential_true).max(), abs(pot_krm).max(), abs(pot_amie).max())
# Clip vmax to avoid KRM blowup dominating the colorbar
vmax_clipped = min(vmax, abs(potential_true).max() * 2)
levels = np.linspace(-vmax_clipped, vmax_clipped, 15)
c = ax.contourf(PHI, r_rad, potential_true, levels=levels, cmap='RdBu_r', extend='both')
ax.set_title('True Potential / 실제 전위', fontsize=11)
plt.colorbar(c, ax=ax, shrink=0.8, label='V')

# KRM-like
ax = axes[1]
c = ax.contourf(PHI, r_rad, np.clip(pot_krm, -vmax_clipped, vmax_clipped),
                levels=levels, cmap='RdBu_r', extend='both')
for ri, pi in obs_locations:
    ax.plot(PHI[ri, pi], r_rad[ri, pi], 'k^', markersize=4)
ax.set_title(f'KRM-like (no $C_a$)\nRMSE={rmse_krm/1000:.1f} kV, '
             f'corr={corr_krm:.3f}', fontsize=10)
plt.colorbar(c, ax=ax, shrink=0.8, label='V')

# AMIE
ax = axes[2]
c = ax.contourf(PHI, r_rad, pot_amie, levels=levels, cmap='RdBu_r', extend='both')
for ri, pi in obs_locations:
    ax.plot(PHI[ri, pi], r_rad[ri, pi], 'k^', markersize=4)
ax.set_title(f'AMIE (with $C_a$)\nRMSE={rmse_amie/1000:.1f} kV, '
             f'corr={corr_amie:.3f}', fontsize=10)
plt.colorbar(c, ax=ax, shrink=0.8, label='V')

for ax in axes:
    ax.set_ylim(0, np.deg2rad(40))
    ax.set_yticks(np.deg2rad([10, 20, 30, 40]))
    ax.set_yticklabels(['80°', '70°', '60°', '50°'], fontsize=8)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(['00', '06', '12', '18'], fontsize=9)

fig.suptitle('KRM vs. AMIE: Effect of $C_a$ Regularization (cf. Fig. 4)\n'
             'KRM vs. AMIE: $C_a$ 정규화의 효과',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n{'Method':<20} {'RMSE (kV)':<12} {'Correlation':<12}")
print(f"{'-'*44}")
print(f"{'KRM-like (no Ca)':<20} {rmse_krm/1000:<12.1f} {corr_krm:<12.4f}")
print(f"{'AMIE (with Ca)':<20} {rmse_amie/1000:<12.1f} {corr_amie:<12.4f}")
print(f"\n→ AMIE's C_a regularization prevents overfitting in data-sparse regions.")
print(f"→ AMIE의 C_a 정규화는 관측이 부족한 지역에서의 과적합을 방지합니다.")

## Part 8: Connection to Modern Data Assimilation / 현대 데이터 동화와의 연결

AMIE's optimal linear estimation is mathematically equivalent to key concepts
in modern data assimilation used in weather forecasting:

| AMIE (1988) | Modern Equivalent / 현대 동등 개념 |
|---|---|
| Optimal interpolation | Kalman filter analysis step |
| $\mathbf{A} = \mathbf{C}_a \mathbf{D}^T (\mathbf{D}\mathbf{C}_a\mathbf{D}^T + \mathbf{C}_v)^{-1}$ | Kalman gain $\mathbf{K} = \mathbf{P}^f \mathbf{H}^T (\mathbf{H}\mathbf{P}^f\mathbf{H}^T + \mathbf{R})^{-1}$ |
| $\mathbf{C}_a$ (fixed) | $\mathbf{P}^f$ (forecast error covariance, evolves) |
| Independent time steps | Sequential assimilation with forecast step |

AMIE의 최적 선형 추정은 현대 기상 예보에서 사용하는 데이터 동화의 핵심 개념과
수학적으로 동등합니다. 이 연결을 시각적으로 보여줍니다.

In [ ]:
# Demonstrate the Kalman filter interpretation of AMIE
# Show how the analysis (posterior) balances prior and observations

# 1D example for clarity: estimate potential at a single point
# as a function of the ratio sigma_obs / sigma_prior

sigma_prior = 15000  # Prior (background) uncertainty in V
sigma_obs_range = np.logspace(2, 5, 100)  # Observation uncertainty range

# True value
x_true = 25000  # True potential at this point
x_prior = 0     # Prior (background) = zero

# Single observation
y_obs = x_true + 3000  # Observation with some noise

# Kalman gain K = sigma_prior^2 / (sigma_prior^2 + sigma_obs^2)
K_values = sigma_prior**2 / (sigma_prior**2 + sigma_obs_range**2)

# Analysis = prior + K * (obs - prior)
x_analysis = x_prior + K_values * (y_obs - x_prior)

# Analysis error
sigma_analysis = np.sqrt((1 - K_values) * sigma_prior**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Kalman gain
ax = axes[0]
ax.semilogx(sigma_obs_range / sigma_prior, K_values, 'b-', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('$\sigma_{obs} / \sigma_{prior}$\n(관측오차 / 배경오차)', fontsize=11)
ax.set_ylabel('Kalman Gain $K$', fontsize=11)
ax.set_title('Kalman Gain\n(= AMIE weight matrix A)', fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.annotate('Trust observation\n관측 신뢰', xy=(0.01, 0.95), fontsize=9, color='green')
ax.annotate('Trust background\n배경 신뢰', xy=(5, 0.05), fontsize=9, color='red')

# Panel 2: Analysis value
ax = axes[1]
ax.semilogx(sigma_obs_range / sigma_prior, x_analysis / 1000, 'r-', linewidth=2,
            label='Analysis / 분석값')
ax.axhline(x_true / 1000, color='green', linestyle='--', label=f'True = {x_true/1000:.0f} kV')
ax.axhline(y_obs / 1000, color='blue', linestyle=':', label=f'Obs = {y_obs/1000:.0f} kV')
ax.axhline(x_prior / 1000, color='gray', linestyle='--', label=f'Prior = {x_prior/1000:.0f} kV')
ax.set_xlabel('$\sigma_{obs} / \sigma_{prior}$', fontsize=11)
ax.set_ylabel('Potential (kV) / 전위', fontsize=11)
ax.set_title('Analysis Value\n(AMIE estimated potential)', fontsize=11)
ax.legend(fontsize=9, loc='center right')

# Panel 3: Analysis error
ax = axes[2]
ax.semilogx(sigma_obs_range / sigma_prior, sigma_analysis / 1000, 'm-', linewidth=2,
            label='Analysis error / 분석 오차')
ax.axhline(sigma_prior / 1000, color='gray', linestyle='--',
           label=f'Prior error = {sigma_prior/1000:.0f} kV')
ax.set_xlabel('$\sigma_{obs} / \sigma_{prior}$', fontsize=11)
ax.set_ylabel('Error std (kV) / 오차 표준편차', fontsize=11)
ax.set_title('Analysis Error\n(cf. Fig. 7 error map)', fontsize=11)
ax.legend(fontsize=9)

fig.suptitle('AMIE = Kalman Filter Analysis Step\n'
             'AMIE는 Kalman filter의 분석 단계와 수학적으로 동일',
             fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

print("Key insight / 핵심 통찰:")
print("• When σ_obs << σ_prior: K → 1, analysis → observation (trust data)")
print("  σ_obs << σ_prior일 때: K → 1, 분석값 → 관측값 (데이터 신뢰)")
print("• When σ_obs >> σ_prior: K → 0, analysis → prior (trust background)")
print("  σ_obs >> σ_prior일 때: K → 0, 분석값 → 배경값 (배경 모델 신뢰)")
print("• Analysis error is ALWAYS less than both prior and observation error")
print("  분석 오차는 항상 배경 오차와 관측 오차 모두보다 작음")

## Summary / 요약

| Part | What we implemented / 구현 내용 | Key concept / 핵심 개념 |
|---|---|---|
| 1 | Basis functions on polar cap / 극관 기저 함수 | $\Phi = \sum a_i \Phi_i$ — all variables share same coefficients / 모든 변수가 동일 계수 공유 |
| 2 | Forward model $\Phi \to E \to J$ / 순방향 모델 | Physical laws connect potential to all observables / 물리 법칙이 전위와 모든 관측 가능량을 연결 |
| 3 | Optimal linear estimation / 최적 선형 추정 | $\hat{a} = C_a D^T(DC_aD^T + C_v)^{-1} \eta$ — Gauss-Markov theorem |
| 4 | Full AMIE workflow / 전체 AMIE 작업 흐름 | Sparse observations → global map with error estimates / 희소 관측 → 전구 지도 + 오차 |
| 5 | Observation density effect / 관측 밀도 효과 | More data → lower error, but only locally / 더 많은 데이터 → 낮은 오차 (지역적) |
| 6 | Conductance modification / 전도도 수정 | Log-space correction keeps $\Sigma > 0$ / 로그 공간 보정으로 전도도 양수 유지 |
| 7 | KRM vs. AMIE comparison / KRM vs. AMIE 비교 | $C_a$ regularization prevents overfitting / $C_a$ 정규화가 과적합 방지 |
| 8 | Modern DA connection / 현대 DA 연결 | AMIE ≡ Kalman filter analysis step / AMIE = Kalman 필터 분석 단계 |

### Next paper / 다음 논문
**#14 Tsyganenko (1989)** — Empirical magnetospheric magnetic field model (T89), which complements AMIE's ionospheric mapping by providing the 3D magnetic field geometry needed for field-line tracing between magnetosphere and ionosphere.

T89 자기권 자기장 모델은 AMIE의 전리층 매핑을 보완하여, 자기권과 전리층 사이의 자기장선 추적에 필요한 3D 자기장 기하학을 제공합니다.